# Echogram Examples

In this notebook, we utilize concatenated data from the Pacific hake survey to create multi-frequency and tricolor echograms. These datasets have been enriched with geographical coordinates.

## Significance of Echogram Visualization

Echogram plotting is of utmost importance for various reasons:

- Fisheries scientists use echograms to scroll through a transect of echosounder data, such as those collected during the hake survey on a ship. This allows them to identify dense aggregations of fish.

- Oceanographers find echograms invaluable for scrolling through months' worth of echosounder data from moorings (e.g., OOI) to observe changes in zooplankton daily movements over extended periods (often 24/7).

- Oceanographers often utilize the "tricolor" echogram, which maps three frequencies to RGB colors. This enables them to distinguish between different fish species based on color variations.

- Fisheries scientists can select a specific area on the echogram display to focus on a region of interest and analyze the `Sv` (volume backscattering strength) within that area for fish analysis. Additionally, they may need to export the dataset sliced by the selected box, allowing them to save the specific `Sv` values within that region to a separate file for further analysis or sharing with colleagues.

## Import Packages and Data 

In [1]:
import holoviews as hv
import panel as pn
import xarray as xr

from echoshader.app import get_box_plot, get_box_stream

`echoshader` works with datasets which are regularly gridded, and the dimensions are `(channel, echo_range/depth, ping_time)`. The `echo_range` should be the same for the whole dataset. We assume they have been produced by the `echopype`'s `compute_MVBS` function, and have the format as displayed below.

In [3]:
from pathlib import Path
from urllib import request

# Calibrated data is stored in Google Drive
url = "https://drive.google.com/uc?export=download&id=197D0MW-bHaF6mZLcQwyr4zqyEHIfwsep"

data_path = Path("concatenated_MVBS.nc")

# Download only if the file is not already present
if not data_path.exists():
    request.urlretrieve(url, data_path)

# Load sample data for testing
MVBS_ds = xr.open_mfdataset(
    paths=data_path,
    data_vars="minimal",
    coords="minimal",
    combine="by_coords",
)

MVBS_ds

<xarray.Dataset> Size: 4MB
Dimensions:            (channel: 4, ping_time: 875, echo_range: 150)
Coordinates:
  * channel            (channel) <U37 592B 'GPT  18 kHz 009072058c8d 1-1 ES18...
  * ping_time          (ping_time) datetime64[ns] 7kB 2017-07-24T19:30:00 ......
    time1              (ping_time) datetime64[ns] 7kB dask.array<chunksize=(875,), meta=np.ndarray>
  * echo_range         (echo_range) float64 1kB 0.0 5.0 10.0 ... 740.0 745.0
Data variables:
    Sv                 (channel, ping_time, echo_range) float64 4MB dask.array<chunksize=(4, 875, 150), meta=np.ndarray>
    frequency_nominal  (channel) float64 32B dask.array<chunksize=(4,), meta=np.ndarray>
    longitude          (ping_time) float64 7kB dask.array<chunksize=(875,), meta=np.ndarray>
    latitude           (ping_time) float64 7kB dask.array<chunksize=(875,), meta=np.ndarray>
Attributes:
    processing_software_name:     echopype
    processing_software_version:  0.7.1
    processing_time:              2023-05-30T17:40:45Z
    processing_function:          commongrid.compute_MVBS

## Echogram Demonstration

Users have the flexibility to personalize echograms by selecting the `channel` to display and applying native HoloViews options to the returned plot:

- `channel`: A list of frequency channels to display as stacks.

- `cmap`: This setting allows users to choose a colormap for the echogram plot through HoloViews `.opts()`. Users can opt for built-in colormaps, such as "jet" (explore the gallery [here](https://holoviews.org/user_guide/Colormaps.html)), or input customized colormaps using arrays, like the "EK500" example shown below.

- `clim`: Set the minimum and maximum values for the Sv (volume backscattering strength) range.


In [4]:
ek500_cmap = [
    "#FFFFFF",
    "#9F9F9F",
    "#5F5F5F",
    "#0000FF",
    "#00007F",
    "#00BF00",
    "#007F00",
    "#FFFF00",
    "#FF7F00",
    "#FF00BF",
    "#FF0000",
    "#A6533C",
    "#783C28",
]

channels = [
    "GPT  18 kHz 009072058c8d 1-1 ES18-11",
    "GPT  38 kHz 009072058146 2-1 ES38B",
    "GPT 120 kHz 00907205a6d0 4-1 ES120-7C",
]

eg = MVBS_ds.eshader.echogram(
    channel=channels,
).opts(
    hv.opts.Image(
        cmap=ek500_cmap,
        clim=(-80, -30),
        colorbar=True,
        width=1250,
        height=450,
    )
)

eg

:Layout
   .Image.GPT_18_kHz_009072058c8d_1_hyphen_minus_1_ES18_hyphen_minus_11   :Image   [ping_time,echo_range]   (Mean volume backscattering strength (MVBS, mean Sv re 1 m-1))
   .Image.GPT_38_kHz_009072058146_2_hyphen_minus_1_ES38B                  :Image   [ping_time,echo_range]   (Mean volume backscattering strength (MVBS, mean Sv re 1 m-1))
   .Image.GPT_120_kHz_00907205a6d0_4_hyphen_minus_1_ES120_hyphen_minus_7C :Image   [ping_time,echo_range]   (Mean volume backscattering strength (MVBS, mean Sv re 1 m-1))

The same controls can be created explicitly with Panel and connected to the echogram:

- `colormap`: This widget allows you to adjust the colormap used for the echograms.

- `Sv_range_slider`: This slider widget enables you to control the Sv (volume backscattering strength) range displayed in the echograms.


In [5]:
Sv_range_slider = pn.widgets.EditableRangeSlider(
    name="Sv Range Slider",
    start=-120,
    end=0,
    value=(-80, -30),
)

colormap = pn.widgets.LiteralInput(
    name="Colormap",
    value="jet",
)

@pn.depends(
    Sv_range_slider.param.value,
    colormap.param.value,
)
def echogram_view(clim, cmap):

    return (
        MVBS_ds.eshader.echogram(
            channel=channels,
        )
        .opts(
            hv.opts.Image(
                cmap=cmap,
                clim=clim,
                colorbar=True,
                tools=["box_select", "lasso_select", "hover"],
                width=1250,
                height=450,
            )
        )
    )

echogram_panel = pn.Row(
    pn.Column(
        Sv_range_slider,
        colormap,
    ),
    echogram_view,
)

echogram_panel


BokehModel(combine_events=True, render_bundle={'docs_json': {'168a732a-c701-4f33-9c75-21de85ce72f7': {'version…

## Tricolor Echogram Demonstration

Tricolor echograms are created with `tricolor_echogram()`.

The order of the channel list determines the RGB mapping relationship. For instance, in the example shown below, the tricolor colormap maps the 120kHz channel to the red channel, the 38kHz channel to the green channel, and the 18Hz channel to the blue channel.


In [11]:
tricolor_eg = MVBS_ds.eshader.tricolor_echogram(
    channel=[
        "GPT 120 kHz 00907205a6d0 4-1 ES120-7C",
        "GPT  38 kHz 009072058146 2-1 ES38B",
        "GPT  18 kHz 009072058c8d 1-1 ES18-11",
    ],
    vmin=-80,
    vmax=-30,
).opts(
    width=1250,
    height=450,
)

pn.Row(tricolor_eg)

BokehModel(combine_events=True, render_bundle={'docs_json': {'65a07060-5ac5-4209-83ac-ea6e8330dbc2': {'version…

A Panel slider can also be connected explicitly to the tricolor echogram:

- `Sv_range_slider`: This slider widget allows you to adjust the Sv (volume backscattering strength) range displayed in the tricolor echograms.


In [13]:
Sv_range_slider = pn.widgets.EditableRangeSlider(
    name="Sv Range Slider",
    start=-120,
    end=0,
    value=(-80, -30),
)

@pn.depends(Sv_range_slider.param.value)
def tricolor_view(clim):
    return (
        MVBS_ds.eshader.tricolor_echogram(
            channel=[
                "GPT 120 kHz 00907205a6d0 4-1 ES120-7C",
                "GPT  38 kHz 009072058146 2-1 ES38B",
                "GPT  18 kHz 009072058c8d 1-1 ES18-11",
            ],
            vmin=clim[0],
            vmax=clim[1],
        )
        .opts(
            width=1250,
            height=450,
        )
    )

tricolor_echogram_panel = pn.Row(
    Sv_range_slider,
    tricolor_view,
)

tricolor_echogram_panel

BokehModel(combine_events=True, render_bundle={'docs_json': {'332abb28-7733-4150-b0ab-10e31c24891d': {'version…

UnknownReferenceError: can't resolve reference '34a40c5a-1fbe-4604-b2d4-0422a102adf1'

UnknownReferenceError: can't resolve reference '34a40c5a-1fbe-4604-b2d4-0422a102adf1'

UnknownReferenceError: can't resolve reference '34a40c5a-1fbe-4604-b2d4-0422a102adf1'

UnknownReferenceError: can't resolve reference '34a40c5a-1fbe-4604-b2d4-0422a102adf1'

## Box Selection

Users can utilize the `Box Select` feature in the toolbar to define a specific rectangular area on the echogram display and obtain the corresponding dataset. The selection stream and overlay are created explicitly with the interaction helpers from `echoshader.app`.

![image.png](./box_select_example.png)

To clear the selected box and reset the box selection, simply use the `Reset Button`.

![image-2.png](./reset_example.png)


In [8]:
box_eg = MVBS_ds.eshader.echogram(
    channel="GPT  38 kHz 009072058146 2-1 ES38B",
).opts(
    cmap="jet",
    clim=(-80, -30),
    colorbar=True,
    tools=["box_select", "hover"],
    width=600,
)

box_stream = get_box_stream(box_eg)
box_overlay = get_box_plot(box_stream)

pn.Row(box_eg * box_overlay)

def data_from_box(ds, bounds, vert_dim="echo_range"):
    if bounds is None:
        return ds

    left, bottom, right, top = bounds

    return ds.sel(
        ping_time=slice(left, right),
        **{vert_dim: slice(min(bottom, top), max(bottom, top))},
    )

data_from_box_select = data_from_box(MVBS_ds, box_stream.bounds)


## Applying Plot Customizations

Users have the option to input `Holoviews options` to tailor the visualizations according to their preferences.

For more in-depth information on using `Holoviews options`, please refer to this [link](https://holoviews.org/user_guide/Applying_Customizations.html#option-list-syntax).

In [20]:
eg = MVBS_ds.eshader.echogram(
    channel="GPT  38 kHz 009072058146 2-1 ES38B",
).opts(
    cmap="Gray",
    clim=(-80, -30),
    width=1250,
    height=450,
)

eg

:Image   [ping_time,echo_range]   (Mean volume backscattering strength (MVBS, mean Sv re 1 m-1))